# NIG-TimeNet (PyTorch) على Colab: دفتر واحد لكل تشغيل مؤتمت

اختر **Runtime ▸ Change runtime type ▸ T4 GPU** ثم **Runtime ▸ Run all**. لا شيء يُلصَق يدوياً ولا خلية تُعدَّل: الخلية الوحيدة التي فيها إعدادات هي التالية.
كل المنطق في `src/nig_timenet_torch/colab/` (واختباراته في `tests/colab/`)؛ الخلايا هنا تستدعيه فقط (القاعدتان E4 وG6).

**الفرق بينه وبين `main.ipynb`:** هذا الدفتر **مُشغِّل مؤتمت**: يقرأ وظيفة جاهزة (`job.json`) من طابور على GitHub، ينفّذها، ويدفع النتائج إلى فرع على GitHub ليقرأها الوكلاء اللاحقون دون أن تنسخ شيئاً.
أما [`main.ipynb`](https://colab.research.google.com/github/yuosef772424/nig-timenet-torch/blob/main/notebooks/main.ipynb) فتفاعلي: ترى كل خطوة وتعدّل إعداداتها بنفسك (وفيه مزامنة Drive وبيئة مخزّنة). استعمل هذا الدفتر للتشغيلات الطويلة المسجَّلة، و`main` للتجريب.

### ما يفعله بالترتيب
1. **الإعداد (Setup):** يربط Drive، ويستنسخ المستودعين **استنساخاً نظيفاً** (`rm -rf` أولاً، فلا يفسده مجلد `/content/tf` متبقٍّ من جلسة سابقة)، ويطبع commit كل منهما، ويثبّت الحزمة **دون المساس بـ torch الخاص بـ Colab**
   (نسخة CUDA الجاهزة؛ التثبيت يُرفض إن تغيّرت نسخة torch)، ثم يتحقق من الجهاز ويسجّله (القاعدة E1: لا «تشغيل GPU» كان في الحقيقة على CPU).
2. **البيانات (Data):** يصدّر بيانات التدريب من مشروع TF (`preprocessing_output_latest.pkl.gz`) على شكل «عقد» (contract artifact) ويتحقق من بصمات ملفاته. يُتخطّى التصدير إن وُجد عقد سليم مصنوع من المصدر نفسه.
3. **المُشغِّل (Runner):** طابور ← تشغيل ← نتائج. الوظيفة هي `job.json` على فرع `colab-queue`؛ ملف النتائج يُكتب على Drive **أولاً** ثم يُدفع إلى فرع `colab-results` (مجلد `runs/<job_id>/`)،
   وخلية `republish` تعيد الدفع إن فشل (القاعدة E2). مخطط `job.json` في `src/nig_timenet_torch/colab/job.py`، ويفحصه `python -m nig_timenet_torch.colab check-job job.json` قبل دفعه.

### أول مرة فقط
ستُسأل عن توكن GitHub (fine-grained، على المستودعين، صلاحية قراءة المحتوى، وقراءة/كتابة على هذا المستودع لدفع النتائج). يُدخَل بحقل مخفي، ويُتحقَّق منه مع GitHub، ويُحفظ في `TOKEN_PATH` على Drive؛
**لا يُطبع ولا يظهر في أي رسالة خطأ ولا يُكتب في `.git/config`**.

In [ ]:
# مستودعا الكود والفرع/الوسم المطلوب لكلٍّ منهما
TORCH_REPO, TORCH_REF = "yuosef772424/nig-timenet-torch", "main"
TF_REPO, TF_REF = "yuosef772424/crypto-signal-prediction", "master"
EXPECT_DEVICE = "cuda"                         # "cuda" أو "cpu": تشغيل cuda على جهاز CPU يفشل عمداً (E1)؛ "cpu" فقط لتشغيل CPU مقصود
# ملف التوكن على Drive (يُنشأ أول مرة)؛ لا يُطبع أبداً
TOKEN_PATH = "/content/drive/MyDrive/crypto/secrets/github_token.txt"
WORKDIR = "/content/nig_runner"                # مجلد عمل مؤقت (سجلات، أداة اعتماد git)؛ لا /tmp أبداً (E2)
TORCH_DIR, TF_DIR = "/content/nig-timenet-torch", "/content/tf"
# حيث تُكتب حزم النتائج على Drive (مجلد لكل وظيفة)
DRIVE_RUNS = "/content/drive/MyDrive/nig_timenet_runs"
# ملف بيانات TF المُعالَجة (المصدر) ثم مسار العقد الناتج
SOURCE = "/content/drive/MyDrive/crypto_model/preprocessed_data/preprocessing_output_latest.pkl.gz"
ARTIFACT = "/content/drive/MyDrive/crypto_model/torch_artifacts/preprocessing_output_1"
ADOPT_EXISTING_ARTIFACT = False                # True: اعتبر عقداً صُنع يدوياً في ARTIFACT مصنوعاً من SOURCE

## 1. Setup — تجهيز الجلسة
الخلية الأولى لا يمكن أن تكون استيراداً عادياً: لا شيء من الحزمة موجود على الجهاز بعد، فهي تنزّل الملف الوحيد الذي لا يعتمد على شيء (`colab/bootstrap.py`) بالتوكن وتنفّذه.
كل ما بعدها استيراد عادي. التالية تستنسخ المستودعين وتثبّت الحزمة، والأخيرة تتحقق من الجهاز وتكتب `setup.json` (commit الكودين، نسخ الحزم، اسم GPU) في `WORKDIR` وعلى Drive.
**ما تتوقعه:** سطر لكل مستودع (`torch: … @ main -> <commit>`) ثم سطر torch/الجهاز؛ إن كتبت `device used cpu` وأنت تتوقع GPU فالتشغيل يفشل هنا عمداً.

In [ ]:
import getpass, os, urllib.error, urllib.request
from google.colab import drive
drive.mount("/content/drive")
token = open(TOKEN_PATH).read().strip() if os.path.exists(TOKEN_PATH) else getpass.getpass("GitHub token (not shown): ")
url = f"https://api.github.com/repos/{TORCH_REPO}/contents/src/nig_timenet_torch/colab/bootstrap.py?ref={TORCH_REF}"
req = urllib.request.Request(url, headers={"Authorization": "Bearer " + token.strip(), "User-Agent": "nig-colab",
                                           "Accept": "application/vnd.github.raw"})
try:
    boot_source = urllib.request.urlopen(req, timeout=60).read().decode()
except urllib.error.HTTPError as e:
    raise SystemExit(f"GitHub answered HTTP {e.code}: the token is missing, expired or cannot read {TORCH_REPO}. "
                     f"Delete {TOKEN_PATH} and run this cell again.")
boot = {}
exec(compile(boot_source, "bootstrap.py", "exec"), boot)
token = boot["acquire_token"](TOKEN_PATH, TORCH_REPO, lambda: getpass.getpass("GitHub token (not shown): "))

In [ ]:
shas = boot["setup_clones"](token, WORKDIR, {"torch": (TORCH_REPO, TORCH_REF, TORCH_DIR),
                                             "tf": (TF_REPO, TF_REF, TF_DIR)})
install = boot["install_package"](TORCH_DIR)

In [ ]:
from nig_timenet_torch.colab.session import check_environment
setup = check_environment(EXPECT_DEVICE, shas, install, WORKDIR, DRIVE_RUNS)

## 2. Data — بيانات TF ← عقد (artifact)
يشغّل `tools/export_torch_artifact.py` من نسخة TF ثم يفحص النتيجة بمحمِّل هذا المشروع (`data_io/artifact.py`): العملات المحجوبة (sealed) مستبعدة، وأحدث شمعة قبل ختم الزمن، وكل ملف يُعاد حساب بصمته.
سجلّ ما صُنع منه العقد هو `<ARTIFACT>.source.json` بجواره؛ يُتخطّى التصدير إن طابق مصدرُ العقد القائم المصدرَ الحالي.
**ما تتوقعه:** سطر ملخّص (الصفوف، العملات، آخر شمعة). **فخ:** `ADOPT_EXISTING_ARTIFACT = True` يعامل عقداً صُنع يدوياً كأنه مصنوع من `SOURCE` — استعمله فقط إن كنت متأكداً.

In [ ]:
from nig_timenet_torch.colab.prepare import prepare_artifact
prep = prepare_artifact(TF_DIR, SOURCE, ARTIFACT, adopt_existing=ADOPT_EXISTING_ARTIFACT)
print(prep["summary"])

## 2b. Grad stack — تجربة التدرجات (اختياري)
النموذج A يتدرّب على النصف الأقدم من بيانات التدريب (ترتيب زمني)، ثم يُخرج توقعاته وتدرّجات كل طبقة على النصف الأحدث وعلى val/test.
النموذج B يتدرّب على النصف الأحدث بمدخل: التدرّجات فقط (`grad`) أو التدرّجات + البيانات (`grad_data`)؛ للمقارنة `data_h2` و`full`.
`grad_full`: النموذج B يرى **فقط** التدرّجات الكاملة للنموذج A (كل طبقة، كل وحدة، قناة لكل مخرج)، بمُرمِّز لكل طبقة؛ لا يرى البيانات ولا توقعات A.
كل النماذج تمرّ على اختبار المشروع (`target_report` + فحوص البوابات). النتائج في `DRIVE_RUNS/grad_stack`.
`grad_target="loss"` يستعمل الهدف الحقيقي فيُسرّب الجواب على test — للمقارنة فقط. حدّث `TORCH_REF` في الخلية الأولى إلى فرع التجربة قبل التشغيل.

In [ ]:
from nig_timenet_torch.workflow.grad_stack import GradStackConfig, grad_stack_battery, run_grad_stack
RUN_GRAD_STACK = True                          # False: تخطَّ هذه التجربة
GRAD_STACK = GradStackConfig(data=ARTIFACT, architecture="", epochs=3, batch_size=256, seed=0, device=EXPECT_DEVICE,
                             max_rows=None, grad_target="output", with_preds=False,
                             full_grad_dim=32, full_grad_outputs="per_head")   # grad_full: التدرّجات الكاملة
if RUN_GRAD_STACK:
    grad_stack = run_grad_stack(GRAD_STACK, f"{DRIVE_RUNS}/grad_stack")
    battery = grad_stack_battery(f"{DRIVE_RUNS}/grad_stack", cost=0.0008)

## 3. Runner — طابور ← تشغيل ← نتائج
يقرأ `job.json` من فرع `colab-queue`، ويرفض وظيفة سبق أن ظهر معرّفها على `colab-results` (إلا إن كتبت فيها `"rerun": true`)، ويسحب الـ commit المثبَّت في `repo_ref`،
ويتحقق من الجهاز المطلوب في الوظيفة، ثم يشغّل المدخل تحت حارس زمن جداري، ويتحقق من المخرجات (بما فيها الجهاز الذي سجّلته الوظيفة نفسها: وظيفة `cuda` سجّلت `cpu` تفشل)،
ويكتب الحزمة على Drive، ويدفعها إلى `colab-results`، ويطبع كتلة حالة.
**ما تتوقعه:** كتلة حالة في النهاية فيها `push: OK` أو `push: FAILED`. المخرجات التي تتجاوز 1 MiB تبقى على Drive ولا تُدفع.

In [ ]:
from nig_timenet_torch.colab.runner import Runner
runner = Runner(f"https://github.com/{TORCH_REPO}.git", token, WORKDIR, DRIVE_RUNS, runner_sha=shas["torch"])
result = runner.run_all()

### Republish — إعادة الدفع
إن قالت كتلة الحالة `push: FAILED` فالحزمة سليمة على Drive (`DRIVE_RUNS/<job_id>`)، وهذه الخلية تدفعها ثانيةً؛ لا تحتاج شيئاً من التشغيل السابق إلا مجلد الحزمة وملف التوكن
(لا `/tmp` ولا نسخة قديمة ولا كائن مُشغِّل — القاعدة E2). في جلسة جديدة بعد خلايا الإعداد: `python -m nig_timenet_torch.colab republish /content/drive/MyDrive/nig_timenet_runs/<job_id>`.

In [ ]:
from nig_timenet_torch.colab.publish import republish
if result["bundle"] and not result["pushed"]:
    republish(result["bundle"], f"https://github.com/{TORCH_REPO}.git", token, WORKDIR)